In [1]:
import torch

# check that model can answer correctly

In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL = "Qwen/Qwen3-8B"

tokenizer = AutoTokenizer.from_pretrained(MODEL)

model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.bfloat16, device_map={"": "cuda:0"})
# bf 16 so that we can fit in memory
# device map so that the weights are copied in GPU

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

In [49]:
prompt = "Answer with only one number. The number of legs on the animal that spins webs is?"
# prompt = "Answer with only one word. The continent where the country that invented paper is located is?"

messages = [
    { "role": "user",
      "content": (prompt),
    }
]

chat_text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False
)
    


input_tokens = tokenizer(chat_text, return_tensors='pt').to(model.device)
print(f'input tokens: {input_tokens}')
with torch.inference_mode():
    output_tokens = model.generate(**input_tokens)

print(f'output tokens: {output_tokens}')

input tokens: {'input_ids': tensor([[151644,    872,    198,  16141,    448,   1172,    825,   1372,     13,
            576,   1372,    315,  14201,    389,    279,   9864,    429,  44758,
          80920,    374,     30, 151645,    198, 151644,  77091,    198, 151667,
            271, 151668,    271]], device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1]], device='cuda:0')}
output tokens: tensor([[151644,    872,    198,  16141,    448,   1172,    825,   1372,     13,
            576,   1372,    315,  14201,    389,    279,   9864,    429,  44758,
          80920,    374,     30, 151645,    198, 151644,  77091,    198, 151667,
            271, 151668,    271,     23, 151645]], device='cuda:0')


/workspace/jacobian-lens-playground/.venv/lib/python3.12/site-packages/transformers/generation/utils.py:1638: UserWarning: Using the model-agnostic default `max_length` (=50) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


In [39]:
last_input_token_id = input_tokens['input_ids'].shape[1]
# the prediction will be from the last token
from_last_token_predictions = output_tokens[0, last_input_token_id:]
print(f'from last token predictions: {from_last_token_predictions}')

from last token predictions: tensor([ 38463, 151645], device='cuda:0')


In [40]:
decoded_response = tokenizer.decode(from_last_token_predictions, skip_special_tokens=True)
print(f'decoded_response: {decoded_response}')

decoded_response: Asia


In [6]:
# j lens

In [41]:
import jlens

jlens_model = jlens.from_hf(model, tokenizer, force_bos=False)

In [42]:
print(jlens_model)

HFLensModel(Qwen3ForCausalLM, n_layers=36, d_model=4096)


In [43]:
jlens_model.layers[-2]

Qwen3DecoderLayer(
  (self_attn): Qwen3Attention(
    (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
    (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
    (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
    (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
    (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
    (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
  )
  (mlp): Qwen3MLP(
    (gate_proj): Linear(in_features=4096, out_features=12288, bias=False)
    (up_proj): Linear(in_features=4096, out_features=12288, bias=False)
    (down_proj): Linear(in_features=12288, out_features=4096, bias=False)
    (act_fn): SiLUActivation()
  )
  (input_layernorm): Qwen3RMSNorm((4096,), eps=1e-06)
  (post_attention_layernorm): Qwen3RMSNorm((4096,), eps=1e-06)
)

# fit jlens

In [44]:
# from prev tests we saw that 

# ------------------
# layer = 32
# ['蜘蛛 spider spiders eight Spider']
# token = ['蜘蛛 spider spiders eight Spider']
# ------------------
# layer = 33
# ['8eight eightEight八']
# token = ['8eight eightEight八']
# ------------------
# layer = 34
# ['8eightEight eight八']
# token = ['8eightEight eight八']

# so we can see from 15 to 30


In [50]:
from jlens.examples import load_wikitext_prompts

layers_to_test = [26, 27, 28, 29, 30, 31]

prompts = load_wikitext_prompts(n_prompts=10)
lens = jlens.fit(jlens_model, prompts, source_layers=layers_to_test)


In [51]:
print(f'chat text : {chat_text}')

chat text : <|im_start|>user
Answer with only one number. The number of legs on the animal that spins webs is?<|im_end|>
<|im_start|>assistant
<think>

</think>




In [52]:
# apply jacobian
lens_logits, _, _ = lens.apply(
    jlens_model, chat_text, layers = layers_to_test, positions = [-1]
)

In [53]:
for layer in layers_to_test:
    print('\n ------------------')
    print(f'layer = {layer}')
    lens_logits_layer = lens_logits[layer]
    _, top_ids = lens_logits_layer.topk(5, dim=-1)
    for id in top_ids[0]:
        print(tokenizer.decode(id), end=",")


 ------------------
layer = 26
蜘蛛, spiders,昆虫,<|endoftext|>,Spider,
 ------------------
layer = 27
蜘蛛, spiders,昆虫, spider,Spider,
 ------------------
layer = 28
蜘蛛, spiders,Spider, spider,昆虫,
 ------------------
layer = 29
蜘蛛, spiders, spider,Spider, Spider,
 ------------------
layer = 30
蜘蛛, spiders, spider, Spider,Spider,
 ------------------
layer = 31
蜘蛛, spiders, spider, Spider,Spider,

# check projections of jacobian vectors on hidden layer

In [89]:
# get spider and ant tokens
spider_strs = ['Spider','蜘蛛']
ant_strs = ['Ant','蚂蚁']

spider_tokens = []
ant_tokens = []

for i in range(2):
    spider_str = spider_strs[i]
    ant_str = ant_strs[i]

    spider_token = tokenizer.encode(spider_str)[0]
    ant_token = tokenizer.encode(ant_str)[0]

    print(f'spider_str = {spider_str}, spider_token = {spider_token}')
    print(f'ant_str = {ant_str}, ant_token = {ant_token}')

    spider_tokens.append(spider_token)
    ant_tokens.append(ant_token)

spider_str = Spider, spider_token = 72908
ant_str = Ant, ant_token = 17117
spider_str = 蜘蛛, spider_token = 111830
ant_str = 蚂蚁, ant_token = 109897


In [90]:
with torch.inference_mode():
    outputs = model.model(**input_tokens, output_hidden_states=True, use_cache=False, return_dict=True)

In [91]:
W_U = model.get_output_embeddings().weight


In [102]:
for layer in layers_to_test:
    h_l = outputs.hidden_states[layer+1][0, -1, :].detach()
    print(f'\n layer = {layer}')
    for i in range(2):
        spider_token = spider_tokens[i]
        ant_token = ant_tokens[i]

        W_U_spider = W_U[spider_token, :]
        W_U_ant = W_U[ant_token, :]

        j_cuda = lens.jacobians[layer].cuda().bfloat16()
        
        spider_j_dir = W_U_spider @ j_cuda
        ant_j_dir = W_U_ant @ j_cuda 
        
        spider_dot_hl = torch.dot(spider_j_dir, h_l)
        ant_dot_hl = torch.dot(ant_j_dir, h_l)
        print(f'spider_dot_hl = {spider_dot_hl}, ant_dot_hl = {ant_dot_hl}, spider > a =ant = {spider_dot_hl > ant_dot_hl}')
    


 layer = 26
spider_dot_hl = 57.75, ant_dot_hl = 3.359375, spider > a =ant = True
spider_dot_hl = 82.0, ant_dot_hl = 49.75, spider > a =ant = True

 layer = 27
spider_dot_hl = 80.0, ant_dot_hl = 22.0, spider > a =ant = True
spider_dot_hl = 111.5, ant_dot_hl = 74.5, spider > a =ant = True

 layer = 28
spider_dot_hl = 93.5, ant_dot_hl = 22.125, spider > a =ant = True
spider_dot_hl = 121.0, ant_dot_hl = 61.25, spider > a =ant = True

 layer = 29
spider_dot_hl = 142.0, ant_dot_hl = 15.375, spider > a =ant = True
spider_dot_hl = 172.0, ant_dot_hl = 74.0, spider > a =ant = True

 layer = 30
spider_dot_hl = 140.0, ant_dot_hl = 21.75, spider > a =ant = True
spider_dot_hl = 179.0, ant_dot_hl = 57.5, spider > a =ant = True

 layer = 31
spider_dot_hl = 124.0, ant_dot_hl = 16.0, spider > a =ant = True
spider_dot_hl = 163.0, ant_dot_hl = 45.0, spider > a =ant = True


# swap


In [123]:
for layer in layers_to_test:
    h_l = outputs.hidden_states[layer+1][0, -1, :].detach()
    print(f'\n layer = {layer}')
    for i in range(2):
        spider_token = spider_tokens[i]
        ant_token = ant_tokens[i]

        W_U_spider = W_U[spider_token, :]
        W_U_ant = W_U[ant_token, :]

        j_cuda = lens.jacobians[layer].cuda().bfloat16()
        
        spider_j_dir = W_U_spider @ j_cuda
        ant_j_dir = W_U_ant @ j_cuda 

        j_dir_stacks = torch.stack([spider_j_dir, ant_j_dir], dim=1)
        j_dir_stack_inv = j_dir_stacks.float().pinverse() # 2 x d

        h_float = h_l.float() # d x 1

        coeffs = j_dir_stack_inv @ h_float # 2 x 1
        swap_coeffs = coeffs.flip(0)

        diff_coeffs = swap_coeffs - coeffs # 2 x 1
        h_new = h_float + diff_coeffs @ j_dir_stacks.float().T

        spider_dot_hl = torch.dot(spider_j_dir, h_l)
        ant_dot_hl = torch.dot(ant_j_dir, h_l)
        print(f'--- token = {spider_tokens[i], ant_tokens[i]} ---')
        print(f'spider_dot_hl = {spider_dot_hl}, ant_dot_hl = {ant_dot_hl}, spider > a =ant = {spider_dot_hl > ant_dot_hl}')

        spider_dot_hnew = torch.dot(spider_j_dir, h_new.bfloat16())
        ant_dot_hnew = torch.dot(ant_j_dir, h_new.bfloat16())
        print(f'** after swapping **')
        print(f'spider_dot_hnew = {spider_dot_hnew}, ant_dot =_hnew = {ant_dot_hnew}, spider > ant = {spider_dot_hnew > ant_dot_hnew}')



        


 layer = 26
--- token = (72908, 17117) ---
spider_dot_hl = 57.75, ant_dot_hl = 3.359375, spider > a =ant = True
** after swapping **
spider_dot_hnew = 1.53125, ant_dot =_hnew = 46.75, spider > ant = False
--- token = (111830, 109897) ---
spider_dot_hl = 82.0, ant_dot_hl = 49.75, spider > a =ant = True
** after swapping **
spider_dot_hnew = 57.75, ant_dot =_hnew = 67.5, spider > ant = False

 layer = 27
--- token = (72908, 17117) ---
spider_dot_hl = 80.0, ant_dot_hl = 22.0, spider > a =ant = True
** after swapping **
spider_dot_hnew = 24.25, ant_dot =_hnew = 66.0, spider > ant = False
--- token = (111830, 109897) ---
spider_dot_hl = 111.5, ant_dot_hl = 74.5, spider > a =ant = True
** after swapping **
spider_dot_hnew = 86.5, ant_dot =_hnew = 93.0, spider > ant = False

 layer = 28
--- token = (72908, 17117) ---
spider_dot_hl = 93.5, ant_dot_hl = 22.125, spider > a =ant = True
** after swapping **
spider_dot_hnew = 24.125, ant_dot =_hnew = 77.5, spider > ant = False
--- token = (111830,

# TODO

 TODO: the swap is correct, but its swapping in pinv(V), it is equal only in some conditions, check what are those conditions?

there is a layer norm in middle of Jacobian and unembedding , what about that ?


SyntaxError: invalid syntax (3192153822.py, line 1)

# run with new h